# Layer correctness: convolution and max-pooling

This notebook validates the forward and backward passes for the core spatial
layers used by the CIFAR-10 CNN.

- Tensors use **NCHW** layout: `(batch, channels, height, width)`.
- Convolution weights use **OIHW** layout: `(out_channels, in_channels, kernel_height, kernel_width)`.
- The convolution operation is cross-correlation, as commonly used in CNNs.
- Finite-difference checks compare analytical gradients against numerical
  gradients on tiny tensors.

In [1]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)
np.random.seed(42)

print("NumPy version:", np.__version__)

NumPy version: 2.2.6


In [2]:
def _pair(value, name):
    """Convert an integer or a pair of integers to a validated pair."""
    if isinstance(value, int):
        pair = (value, value)
    elif isinstance(value, tuple) and len(value) == 2:
        pair = value
    else:
        raise TypeError(f"{name} must be an int or a pair of ints")

    if any(not isinstance(item, int) or item <= 0 for item in pair):
        raise ValueError(f"{name} values must be positive integers")

    return pair


def _padding_for_conv(height, width, kernel_height, kernel_width, stride, padding):
    """Return top, bottom, left, and right convolution padding."""
    if isinstance(padding, str):
        if padding == "valid":
            return 0, 0, 0, 0
        if padding == "same":
            output_height = (height + stride - 1) // stride
            output_width = (width + stride - 1) // stride

            total_height = max(
                (output_height - 1) * stride + kernel_height - height,
                0,
            )
            total_width = max(
                (output_width - 1) * stride + kernel_width - width,
                0,
            )

            top = total_height // 2
            left = total_width // 2
            return top, total_height - top, left, total_width - left

        raise ValueError("padding must be 'valid', 'same', an int, or a pair")

    if isinstance(padding, int):
        if padding < 0:
            raise ValueError("padding cannot be negative")
        return padding, padding, padding, padding

    if isinstance(padding, tuple) and len(padding) == 2:
        pad_height, pad_width = padding
        if any(not isinstance(item, int) or item < 0 for item in padding):
            raise ValueError("padding values must be non-negative integers")
        return pad_height, pad_height, pad_width, pad_width

    raise TypeError("padding must be 'valid', 'same', an int, or a pair")


def conv2d_forward(x, weights, bias, stride=1, padding="valid"):
    """
    Apply a 2D multi-channel cross-correlation.

    Args:
        x: Input tensor with shape (N, C_in, H, W).
        weights: Filter tensor with shape (C_out, C_in, K_h, K_w).
        bias: Bias vector with shape (C_out,).
        stride: Positive integer stride.
        padding: "valid", "same", an integer, or a (height, width) pair.

    Returns:
        Output tensor with shape (N, C_out, H_out, W_out), and a cache
        required by conv2d_backward.
    """
    x = np.asarray(x, dtype=np.float64)
    weights = np.asarray(weights, dtype=np.float64)
    bias = np.asarray(bias, dtype=np.float64)

    if x.ndim != 4 or weights.ndim != 4:
        raise ValueError("x and weights must both be rank-4 tensors")

    if not isinstance(stride, int) or stride <= 0:
        raise ValueError("stride must be a positive integer")

    batch_size, in_channels, height, width = x.shape
    out_channels, weight_in_channels, kernel_height, kernel_width = weights.shape

    if in_channels != weight_in_channels:
        raise ValueError("Input channels must match the weight input channels")
    if bias.shape != (out_channels,):
        raise ValueError(f"bias must have shape ({out_channels},)")

    pad_top, pad_bottom, pad_left, pad_right = _padding_for_conv(
        height,
        width,
        kernel_height,
        kernel_width,
        stride,
        padding,
    )

    x_padded = np.pad(
        x,
        (
            (0, 0),
            (0, 0),
            (pad_top, pad_bottom),
            (pad_left, pad_right),
        ),
    )

    padded_height, padded_width = x_padded.shape[2:]
    output_height = (padded_height - kernel_height) // stride + 1
    output_width = (padded_width - kernel_width) // stride + 1

    if output_height <= 0 or output_width <= 0:
        raise ValueError("Kernel is larger than the padded input")

    output = np.empty(
        (batch_size, out_channels, output_height, output_width),
        dtype=np.float64,
    )

    for out_y in range(output_height):
        y_start = out_y * stride
        y_end = y_start + kernel_height

        for out_x in range(output_width):
            x_start = out_x * stride
            x_end = x_start + kernel_width
            patch = x_padded[:, :, y_start:y_end, x_start:x_end]
            output[:, :, out_y, out_x] = (
                np.einsum("nchw,ochw->no", patch, weights) + bias
            )

    cache = {
        "x_padded": x_padded,
        "weights": weights,
        "input_shape": x.shape,
        "stride": stride,
        "pad_top": pad_top,
        "pad_left": pad_left,
    }
    return output, cache

In [3]:
def conv2d_backward(d_output, cache):
    """
    Backpropagate through conv2d_forward.

    Returns gradients for the input, weights, and bias.
    """
    d_output = np.asarray(d_output, dtype=np.float64)

    x_padded = cache["x_padded"]
    weights = cache["weights"]
    input_shape = cache["input_shape"]
    stride = cache["stride"]
    pad_top = cache["pad_top"]
    pad_left = cache["pad_left"]

    batch_size, in_channels, height, width = input_shape
    out_channels, _, kernel_height, kernel_width = weights.shape
    _, _, output_height, output_width = d_output.shape

    expected_shape = (batch_size, out_channels, output_height, output_width)
    if d_output.shape != expected_shape:
        raise ValueError("d_output has an incompatible shape")

    d_x_padded = np.zeros_like(x_padded)
    d_weights = np.zeros_like(weights)
    d_bias = np.sum(d_output, axis=(0, 2, 3))

    for out_y in range(output_height):
        y_start = out_y * stride
        y_end = y_start + kernel_height

        for out_x in range(output_width):
            x_start = out_x * stride
            x_end = x_start + kernel_width
            patch = x_padded[:, :, y_start:y_end, x_start:x_end]
            grad = d_output[:, :, out_y, out_x]

            d_weights += np.einsum("no,nchw->ochw", grad, patch)
            d_x_padded[:, :, y_start:y_end, x_start:x_end] += np.einsum(
                "no,ochw->nchw",
                grad,
                weights,
            )

    d_x = d_x_padded[
        :,
        :,
        pad_top : pad_top + height,
        pad_left : pad_left + width,
    ]
    return d_x, d_weights, d_bias

In [4]:
x_example = np.arange(1, 1 + 1 * 2 * 4 * 4, dtype=np.float64).reshape(1, 2, 4, 4)
weights_example = np.ones((3, 2, 3, 3), dtype=np.float64)
bias_example = np.zeros(3, dtype=np.float64)

y_valid, _ = conv2d_forward(
    x_example,
    weights_example,
    bias_example,
    stride=1,
    padding="valid",
)
y_same, _ = conv2d_forward(
    x_example,
    weights_example,
    bias_example,
    stride=1,
    padding="same",
)

assert y_valid.shape == (1, 3, 2, 2)
assert y_same.shape == (1, 3, 4, 4)

# All three output channels use the same filters, so their outputs match.
np.testing.assert_allclose(y_valid[:, 0], y_valid[:, 1])
np.testing.assert_allclose(y_valid[:, 1], y_valid[:, 2])

print("Valid output shape:", y_valid.shape)
print("Same output shape:", y_same.shape)
print("Multi-channel convolution shape and values: OK")

Valid output shape: (1, 3, 2, 2)
Same output shape: (1, 3, 4, 4)
Multi-channel convolution shape and values: OK


In [5]:
def numerical_gradient(array, objective, epsilon=1e-5):
    """Calculate a central finite-difference gradient in place safely."""
    gradient = np.zeros_like(array, dtype=np.float64)
    iterator = np.nditer(array, flags=["multi_index"], op_flags=["readwrite"])

    while not iterator.finished:
        index = iterator.multi_index
        original_value = array[index]

        array[index] = original_value + epsilon
        objective_plus = objective()

        array[index] = original_value - epsilon
        objective_minus = objective()

        array[index] = original_value
        gradient[index] = (objective_plus - objective_minus) / (2 * epsilon)
        iterator.iternext()

    return gradient


def relative_error(analytical, numerical):
    numerator = np.linalg.norm(analytical - numerical)
    denominator = max(
        1e-12,
        np.linalg.norm(analytical) + np.linalg.norm(numerical),
    )
    return numerator / denominator


rng = np.random.default_rng(7)
x = rng.normal(size=(1, 2, 4, 4))
weights = rng.normal(size=(2, 2, 3, 3))
bias = rng.normal(size=(2,))

output, cache = conv2d_forward(x, weights, bias, stride=1, padding="same")
upstream = rng.normal(size=output.shape)

d_x, d_weights, d_bias = conv2d_backward(upstream, cache)

def conv_objective():
    result, _ = conv2d_forward(x, weights, bias, stride=1, padding="same")
    return np.sum(result * upstream)

numeric_d_x = numerical_gradient(x, conv_objective)
numeric_d_weights = numerical_gradient(weights, conv_objective)
numeric_d_bias = numerical_gradient(bias, conv_objective)

errors = {
    "input": relative_error(d_x, numeric_d_x),
    "weights": relative_error(d_weights, numeric_d_weights),
    "bias": relative_error(d_bias, numeric_d_bias),
}

for name, error in errors.items():
    print(f"Convolution {name} gradient relative error: {error:.3e}")
    assert error < 1e-7, f"Convolution {name} gradient check failed"

print("Convolution gradient checks: OK")

Convolution input gradient relative error: 1.601e-11
Convolution weights gradient relative error: 2.398e-11
Convolution bias gradient relative error: 1.926e-11
Convolution gradient checks: OK


In [6]:
def max_pool2d_forward(x, pool_size=2, stride=None):
    """
    Apply valid max-pooling to an NCHW tensor.

    For tied maxima, the backward pass distributes the gradient evenly
    among all tied maximum positions in the window.
    """
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 4:
        raise ValueError("x must be a rank-4 NCHW tensor")

    pool_height, pool_width = _pair(pool_size, "pool_size")
    if stride is None:
        stride = pool_height
    if not isinstance(stride, int) or stride <= 0:
        raise ValueError("stride must be a positive integer")

    batch_size, channels, height, width = x.shape
    output_height = (height - pool_height) // stride + 1
    output_width = (width - pool_width) // stride + 1

    if output_height <= 0 or output_width <= 0:
        raise ValueError("Pooling window is larger than the input")

    output = np.empty(
        (batch_size, channels, output_height, output_width),
        dtype=np.float64,
    )
    max_mask = np.zeros_like(x, dtype=bool)

    for out_y in range(output_height):
        y_start = out_y * stride
        y_end = y_start + pool_height

        for out_x in range(output_width):
            x_start = out_x * stride
            x_end = x_start + pool_width
            window = x[:, :, y_start:y_end, x_start:x_end]
            maxima = np.max(window, axis=(2, 3), keepdims=True)

            output[:, :, out_y, out_x] = maxima[:, :, 0, 0]
            max_mask[:, :, y_start:y_end, x_start:x_end] |= window == maxima

    cache = {
        "input_shape": x.shape,
        "max_mask": max_mask,
        "pool_size": (pool_height, pool_width),
        "stride": stride,
        "output_shape": output.shape,
    }
    return output, cache


def max_pool2d_backward(d_output, cache):
    """Backpropagate through max_pool2d_forward."""
    d_output = np.asarray(d_output, dtype=np.float64)

    input_shape = cache["input_shape"]
    max_mask = cache["max_mask"]
    pool_height, pool_width = cache["pool_size"]
    stride = cache["stride"]

    if d_output.shape != cache["output_shape"]:
        raise ValueError("d_output has an incompatible shape")

    batch_size, channels, _, _ = input_shape
    _, _, output_height, output_width = d_output.shape
    d_x = np.zeros(input_shape, dtype=np.float64)

    for out_y in range(output_height):
        y_start = out_y * stride
        y_end = y_start + pool_height

        for out_x in range(output_width):
            x_start = out_x * stride
            x_end = x_start + pool_width
            mask = max_mask[:, :, y_start:y_end, x_start:x_end]

            # Count tied maxima separately for each batch item and channel.
            counts = np.sum(mask, axis=(2, 3), keepdims=True)
            window_grad = (
                mask
                * d_output[:, :, out_y, out_x, None, None]
                / counts
            )
            d_x[:, :, y_start:y_end, x_start:x_end] += window_grad

    return d_x

In [7]:
x_pool = np.array(
    [[[[1.0, 2.0, 3.0],
       [4.0, 9.0, 6.0],
       [7.0, 8.0, 5.0]]]]
)

pooled, pool_cache = max_pool2d_forward(x_pool, pool_size=2, stride=1)
np.testing.assert_allclose(pooled, np.array([[[[9.0, 9.0], [9.0, 9.0]]]]))

pool_upstream = np.array([[[[1.0, 2.0], [3.0, 4.0]]]])
pool_d_x = max_pool2d_backward(pool_upstream, pool_cache)

expected_pool_d_x = np.zeros_like(x_pool)
expected_pool_d_x[0, 0, 1, 1] = 10.0
np.testing.assert_allclose(pool_d_x, expected_pool_d_x)

def pool_objective():
    result, _ = max_pool2d_forward(x_pool, pool_size=2, stride=1)
    return np.sum(result * pool_upstream)

numeric_pool_d_x = numerical_gradient(x_pool, pool_objective)
pool_error = relative_error(pool_d_x, numeric_pool_d_x)

print("Max-pool output:\n", pooled)
print("Max-pool input gradient:\n", pool_d_x)
print(f"Max-pooling input gradient relative error: {pool_error:.3e}")

assert pool_error < 1e-7, "Max-pooling gradient check failed"
print("Max-pooling forward and gradient checks: OK")

Max-pool output:
 [[[[9. 9.]
   [9. 9.]]]]
Max-pool input gradient:
 [[[[ 0.  0.  0.]
   [ 0. 10.  0.]
   [ 0.  0.  0.]]]]
Max-pooling input gradient relative error: 1.660e-11
Max-pooling forward and gradient checks: OK


In [8]:
print("All layer correctness checks passed.")
print("Validated: multi-channel convolution, padding, output shapes,")
print("convolution gradients, overlapping max-pooling, and pooling gradients.")

All layer correctness checks passed.
Validated: multi-channel convolution, padding, output shapes,
convolution gradients, overlapping max-pooling, and pooling gradients.
